# Person 1 — Data Collection & Preparation
**Programming for AI | TMDB Movie Data Project** · Data source: The Movie Database (TMDB) · Period: 2016–2026

**Scope of this notebook (Person 1):** TMDB account & credentials → API collection (Discover + Details + Credits) → pagination & ID extraction → raw CSV → collection log + SHA-256 fingerprint → data understanding → missing-value analysis → duplicate handling → controlled 5% missingness + imputation → **Advanced technique #1: KNN / Iterative imputation**.

**Prediction target chosen for the project:** `revenue` (box-office revenue, USD). This is a numerical target, so it suits regression. It is decided here because the missing-value rules depend on the target. Person 2 can still change it, but must tell Person 1 first.

**Outputs written to `data/`:**

| File | Purpose |
|---|---|
| `movie_ids.csv` | collected IDs with year and discover rank |
| `raw_responses.jsonl` | raw Details+Credits API responses (reproducibility) |
| `tmdb_movies_raw.csv` | flattened, **unmodified** dataset straight from the API |
| `collection_log.json`, `sha256_manifest.json` | collection record and file fingerprints |
| `tmdb_movies_final.csv` | duplicates removed, 0→NaN for budget/revenue/runtime |
| `tmdb_movies_model_base.csv` | final CSV minus rows with missing target |
| `tmdb_movies_working_missing.csv` | working copy with ~5% artificial missingness |
| `tmdb_movies_imputed.csv` | working copy after imputation |

**How to run:** set your token with `export TMDB_TOKEN="..."` (or paste it when prompted), then *Run All*. Collection takes roughly 10–20 minutes and is resumable, because already-downloaded movies are cached in `data/raw_json/`.

In [1]:
import os, json, time, hashlib, datetime as dt, warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

SEED = 42
DATA_DIR = "data"
RAW_DIR = os.path.join(DATA_DIR, "raw_json")
FIG_DIR = "figures"
for d in (DATA_DIR, RAW_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

RAW_CSV = os.path.join(DATA_DIR, "tmdb_movies_raw.csv")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("Setup complete. Python libs loaded.")

Matplotlib is building the font cache; this may take a moment.


Setup complete. Python libs loaded.


**What was done:** imported the libraries, fixed the random seed (`SEED = 42`) and created the `data/` and `figures/` folders.
**Why:** a fixed seed makes the artificial missingness and every random step reproducible, and separate folders keep the deliverables organised.
**Output:** only a confirmation message is printed.
**Observation:** nothing is analysed yet. Every later random operation uses this seed, so a teammate re-running the notebook gets identical results.

## Step 1 — TMDB account and API credentials
1. Create a free account at https://www.themoviedb.org/ and verify the email.
2. Go to **Settings → API** and request a Developer key (select *Education / Personal use*).
3. Copy the **API Read Access Token** (the long JWT string), not the short v3 key. It is sent as a `Bearer` token.
4. Never hard-code the token in the notebook. Store it in an environment variable (`TMDB_TOKEN`) or paste it at runtime, so it is not leaked when the notebook is submitted or pushed to GitHub.

TMDB requires attribution (*"This product uses the TMDB API but is not endorsed or certified by TMDB"*) and applies rate limits (roughly 40–50 requests/second). The collector below sleeps between calls and retries on HTTP 429 to stay within these terms.

In [3]:
from getpass import getpass

TOKEN = os.environ.get("TMDB_TOKEN") or getpass("Paste your TMDB API Read Access Token: ")
BASE = "https://api.themoviedb.org/3"

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {TOKEN}", "accept": "application/json"})

def tmdb_get(path, params=None, retries=6):
    """GET wrapper: retries on 429 (rate limit) and 5xx with back-off."""
    for attempt in range(retries):
        r = session.get(f"{BASE}{path}", params=params, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            time.sleep(int(r.headers.get("Retry-After", 2 ** attempt)))
            continue
        if r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
    raise RuntimeError(f"Failed after {retries} retries: {path}")

cfg = tmdb_get("/configuration")          # cheap call to prove the credentials work
print("Authentication OK. Image base URL:", cfg["images"]["base_url"])

Paste your TMDB API Read Access Token:  ········


Authentication OK. Image base URL: http://image.tmdb.org/t/p/


**What was done:** read the token securely, built a `requests.Session` with the Bearer header, and wrote `tmdb_get()`, a wrapper that retries on rate-limit (429) and server errors (5xx). A call to `/configuration` verifies the credentials.
**Why:** one shared, fault-tolerant request function keeps the collection code short and stops a temporary API hiccup from killing a 1,000+ request job.
**Output:** "Authentication OK" plus the image base URL.
**Observation:** a successful response means the account and token are valid. A 401 error would mean the wrong token type was pasted (the v3 key was used instead of the Read Access Token).

## Step 2 & 3 — Discover endpoint: yearly collection, pagination and movie-ID extraction

**Design decisions (to be documented in the final report):**

| Decision | Choice | Justification |
|---|---|---|
| Endpoint | `/discover/movie` | Required by the brief; supports date filters and sorting |
| Date filter | `primary_release_date.gte/lte` = 1 Jan – 31 Dec of each year | Retrieves one calendar year at a time, as required |
| Sort criterion | `vote_count.desc` | Films with many votes almost always have complete metadata (budget, revenue, runtime, credits). Sorting by popularity would favour only currently trending titles and be unstable between runs. |
| Page size | 20 results per page (fixed by TMDB) | About 5 pages per year are needed for 100 movies |
| Target size | 100 unique movies per year | About 1,100 movies in total. 2026 is partial, so the actual count is reported. |
| Adult content | `include_adult=false` | Not relevant to the project |
| De-duplication | A global `seen` set | The same ID is never added twice, and no records are created artificially |

*Known limitation:* sorting by vote count introduces **popularity bias**, because obscure films are under-represented. This must be mentioned in the limitations section. Budget is **not** a sort option in Discover, so budget variation is retained naturally. Records are not filtered by budget.

In [4]:
START_YEAR, END_YEAR = 2016, 2026
TARGET_PER_YEAR = 100
SORT_BY = "vote_count.desc"
MAX_PAGES = 500                      # TMDB hard limit on pagination

collection_started = dt.datetime.now().isoformat(timespec="seconds")
seen, id_records, per_year_log = set(), [], []

for year in range(START_YEAR, END_YEAR + 1):
    collected, page, pages_fetched = [], 1, 0
    total_pages = total_results = None
    while len(collected) < TARGET_PER_YEAR:
        params = {
            "primary_release_date.gte": f"{year}-01-01",
            "primary_release_date.lte": f"{year}-12-31",
            "sort_by": SORT_BY, "include_adult": "false",
            "language": "en-US", "page": page,
        }
        data = tmdb_get("/discover/movie", params)
        total_pages, total_results = data["total_pages"], data["total_results"]
        pages_fetched += 1
        for m in data["results"]:
            if m["id"] not in seen and len(collected) < TARGET_PER_YEAR:
                seen.add(m["id"])
                collected.append(m["id"])
                id_records.append({"id": m["id"], "year": year, "discover_rank": len(collected),
                                   "discover_page": page, "vote_count_at_discovery": m.get("vote_count")})
        if page >= min(total_pages, MAX_PAGES) or not data["results"]:
            break
        page += 1
        time.sleep(0.05)
    per_year_log.append({"year": year, "api_total_results": total_results, "api_total_pages": total_pages,
                         "pages_fetched": pages_fetched, "ids_collected": len(collected)})
    print(f"{year}: collected {len(collected):>3} IDs  (API reports {total_results} results / {total_pages} pages; fetched {pages_fetched} pages)")

ids_df = pd.DataFrame(id_records)
ids_df.to_csv(os.path.join(DATA_DIR, "movie_ids.csv"), index=False)
all_ids = ids_df["id"].tolist()
print("\nTotal unique IDs:", len(all_ids), "| duplicates in list:", len(all_ids) - len(set(all_ids)))

2016: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 5 pages)
2017: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 5 pages)
2018: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 5 pages)
2019: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 5 pages)
2020: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 5 pages)
2021: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 5 pages)
2022: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 5 pages)
2023: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 5 pages)
2024: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 5 pages)
2025: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 5 pages)
2026: collected 100 IDs  (API reports 20001 results / 1001 pages; fetched 6 pages)

Total unique IDs: 1100 | duplicates in list: 0


**What was done:** looped over 2016–2026, calling Discover with the year's date range, sorted by `vote_count.desc`. It paged through results until 100 unique IDs were collected, or the API ran out of results. Every ID, its year, its rank and the page it came from were logged.
**Why:** the Discover endpoint is the only way to filter by date. Paging and the global `seen` set guarantee unique IDs without padding the dataset artificially. `movie_ids.csv` is the required record of collected identifiers.
**Output:** one line per year (IDs collected, API total results, pages fetched), then the total unique ID count.
**Observation:** *(complete after running)* — state the total ID count, confirm 2016–2025 reached about 100 each, and give the actual count for 2026 (partial year, so it may be below 100). The duplicate count should be 0.

## Step 4 — Movie Details + Credits retrieval
For each ID we call `/movie/{id}?append_to_response=credits`. This returns the details (budget, revenue, runtime, genres, companies, countries, languages, etc.) **and** the cast/crew in a single request, which halves the number of API calls. Each raw JSON response is saved to disk, so an interrupted run resumes without re-downloading.

In [ ]:
failed = []
t0 = time.time()
for i, mid in enumerate(all_ids, 1):
    fp = os.path.join(RAW_DIR, f"{mid}.json")
    if os.path.exists(fp):
        continue
    try:
        d = tmdb_get(f"/movie/{mid}", {"append_to_response": "credits", "language": "en-US"})
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(d, f, ensure_ascii=False)
    except Exception as e:
        failed.append((mid, str(e)))
    if i % 100 == 0:
        print(f"{i}/{len(all_ids)} done ({time.time()-t0:.0f}s)")
    time.sleep(0.03)

print("Failed IDs:", failed if failed else "none")

# combine all raw responses into one JSONL file (raw-data copy for reproducibility)
raw_jsonl = os.path.join(DATA_DIR, "raw_responses.jsonl")
n_written = 0
with open(raw_jsonl, "w", encoding="utf-8") as out:
    for mid in all_ids:
        fp = os.path.join(RAW_DIR, f"{mid}.json")
        if os.path.exists(fp):
            out.write(json.dumps(json.load(open(fp, encoding="utf-8")), ensure_ascii=False) + "\n")
            n_written += 1
print("Raw responses saved:", n_written)

100/1100 done (36s)
200/1100 done (72s)


**What was done:** downloaded the details and credits of every collected ID, cached each response as JSON, and merged all responses into `raw_responses.jsonl`.
**Why:** the Discover response does not contain budget, revenue, runtime or credits. Keeping the raw JSON means the CSV can be rebuilt without calling the API again, which makes the project reproducible.
**Output:** progress messages, the list of failed IDs, and the number of raw responses saved.
**Observation:** *(complete after running)* — a failed list of "none" means all requests succeeded. If some IDs failed, re-run the cell; cached IDs are skipped.

In [ ]:
def names(lst, key="name"):
    return "|".join(x[key] for x in lst) if lst else None

def build_row(d):
    credits = d.get("credits") or {}
    crew, cast = credits.get("crew") or [], credits.get("cast") or []
    genres, comps = d.get("genres") or [], d.get("production_companies") or []
    countries, langs = d.get("production_countries") or [], d.get("spoken_languages") or []
    coll = d.get("belongs_to_collection")
    director = next((p["name"] for p in crew if p.get("job") == "Director"), None)
    rd = d.get("release_date") or None
    return {
        "id": d["id"], "imdb_id": d.get("imdb_id"),
        "title": d.get("title"), "original_title": d.get("original_title"),
        "release_date": rd, "status": d.get("status"),
        "original_language": d.get("original_language"),
        "budget": d.get("budget"), "revenue": d.get("revenue"), "runtime": d.get("runtime"),
        "popularity": d.get("popularity"), "vote_average": d.get("vote_average"), "vote_count": d.get("vote_count"),
        "adult": d.get("adult"), "video": d.get("video"),
        "has_homepage": bool(d.get("homepage")), "has_tagline": bool(d.get("tagline")),
        "in_collection": coll is not None, "collection_name": coll["name"] if coll else None,
        "genres": names(genres), "n_genres": len(genres), "primary_genre": genres[0]["name"] if genres else None,
        "production_companies": names(comps), "n_companies": len(comps),
        "primary_company": comps[0]["name"] if comps else None,
        "production_countries": names(countries), "n_countries": len(countries),
        "primary_country": countries[0]["name"] if countries else None,
        "n_spoken_languages": len(langs),
        "director": director, "top_cast": "|".join(p["name"] for p in cast[:3]) or None,
        "cast_size": len(cast), "crew_size": len(crew),
        "overview": d.get("overview") or None, "tagline": d.get("tagline") or None,
    }

rows = []
with open(raw_jsonl, encoding="utf-8") as f:
    for line in f:
        rows.append(build_row(json.loads(line)))
df_build = pd.DataFrame(rows)

# date-derived attributes
rd = pd.to_datetime(df_build["release_date"], errors="coerce")
df_build["release_year"] = rd.dt.year
df_build["release_month"] = rd.dt.month
df_build["release_dow"] = rd.dt.dayofweek          # 0 = Monday

df_build.to_csv(RAW_CSV, index=False)
print("Saved", RAW_CSV, "| shape:", df_build.shape)
df_build.head(3)

**What was done:** flattened the nested JSON into one row per movie. Multi-valued fields (genres, companies, countries, top-3 cast) are stored as `|`-separated strings, and counts (`n_genres`, `n_companies`, `cast_size`, …) and "primary" values (first genre, first company, first country, director) are extracted. Date-derived columns (`release_year`, `release_month`, `release_dow`) and Boolean flags (`has_homepage`, `has_tagline`, `in_collection`) were created. The result was saved as `tmdb_movies_raw.csv` **without any cleaning**.
**Why:** the brief asks for numerical, categorical, date-derived and Boolean attributes in a single table with one movie per row. IDs and titles are kept for identification only and must not be used as numeric predictors. The raw CSV is the "original unmodified dataset" deliverable. Budget/revenue/runtime zeros are kept exactly as TMDB returned them.
**Output:** file path, shape, and the first three rows.
**Observation:** *(complete after running)* — note the number of rows and columns, and that budget/revenue/runtime still contain 0 values at this stage.

## Step 5 — Data-collection log and SHA-256 fingerprint
A SHA-256 hash is a unique "fingerprint" of a file. If even one byte changes, the hash changes. Recording it proves that the raw dataset submitted is the same one that was collected, and lets the instructor verify integrity.

In [ ]:
ids_fingerprint = hashlib.sha256(",".join(map(str, sorted(all_ids))).encode()).hexdigest()

log = {
    "collection_started": collection_started,
    "collection_finished": dt.datetime.now().isoformat(timespec="seconds"),
    "api": "TMDB v3", "base_url": BASE,
    "endpoints": ["/discover/movie", "/movie/{id}?append_to_response=credits"],
    "date_filter": f"primary_release_date {START_YEAR}-01-01 to {END_YEAR}-12-31 (one calendar year per query)",
    "sort_by": SORT_BY, "include_adult": False, "language": "en-US",
    "target_per_year": TARGET_PER_YEAR, "pagination": "20 results/page, pages fetched until target reached",
    "per_year": per_year_log, "total_unique_ids": len(all_ids),
    "details_failed": failed, "movie_ids_sha256": ids_fingerprint,
    "python_requests_version": requests.__version__, "pandas_version": pd.__version__,
}
with open(os.path.join(DATA_DIR, "collection_log.json"), "w") as f:
    json.dump(log, f, indent=2)

manifest = {p: sha256_file(os.path.join(DATA_DIR, p)) for p in ["tmdb_movies_raw.csv", "raw_responses.jsonl", "movie_ids.csv"]}
with open(os.path.join(DATA_DIR, "sha256_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print("Collection log saved.\nSHA-256 fingerprints:")
for k, v in manifest.items():
    print(f"  {k:<24} {v}")
print("  movie IDs list           ", ids_fingerprint)
pd.DataFrame(per_year_log)

**What was done:** wrote `collection_log.json` (dates, endpoints, filters, sort criterion, pagination, per-year counts, failures) and computed SHA-256 hashes for the raw CSV, raw JSONL, ID file and the sorted ID list.
**Why:** the brief requires a record of the API approach (date filters, sort, pagination, IDs). The hashes make the raw data tamper-evident.
**Output:** the fingerprints and the per-year collection table.
**Observation:** *(complete after running)* — copy the raw CSV hash into the report. Any later change to `tmdb_movies_raw.csv` will change the hash, so the raw file must never be edited; all cleaning happens on copies.

# PHASE 2 — Data understanding and preparation
*From here on the notebook only needs `data/tmdb_movies_raw.csv`, so it can be re-run without the API.*
## Step 6 — Dataset understanding and description

In [ ]:
# [PHASE2-START]
df_raw = pd.read_csv(RAW_CSV)
df_raw["release_date"] = pd.to_datetime(df_raw["release_date"], errors="coerce")
df = df_raw.copy()

print("First 5 records"); display(df.head())
print("Last 5 records");  display(df.tail())
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
print("\nColumns:", list(df.columns))

**What was done:** loaded the raw CSV, then displayed the first and last five records, the number of rows and columns, and the column names.
**Why:** this is the first sanity check. It confirms the CSV matches what was collected and shows the layout of the attributes.
**Output:** two preview tables, the shape and the column list.
**Observation:** *(complete after running)* — the first rows should be 2016 films with the highest vote counts and the last rows 2026 films with low counts. Note any obviously suspicious values, such as `0` budgets or missing taglines.

In [ ]:
df.info()
print("\nMemory usage (deep): %.2f MB" % (df.memory_usage(deep=True).sum() / 1024**2))

**What was done:** ran `df.info()` (data types and non-null counts) and measured memory usage.
**Why:** to check that every column has the expected data type (dates parsed as datetime, flags as bool, counts as integers), and to confirm the dataset is small enough that memory is not a concern.
**Output:** a column-by-column table of non-null counts and dtypes, followed by memory in MB.
**Observation:** *(complete after running)* — mention which columns have fewer non-null values than the row count (candidates are `tagline`, `collection_name`, `director`, `overview`) and give the memory figure.

In [ ]:
ID_COLS   = ["id", "imdb_id"]
TEXT_COLS = ["title", "original_title", "overview", "tagline"]
DATE_COLS = ["release_date"]
BOOL_COLS = ["adult", "video", "has_homepage", "has_tagline", "in_collection"]
NUM_COLS  = ["budget", "revenue", "runtime", "popularity", "vote_average", "vote_count",
             "n_genres", "n_companies", "n_countries", "n_spoken_languages", "cast_size", "crew_size",
             "release_year", "release_month", "release_dow"]
CAT_COLS  = ["status", "original_language", "primary_genre", "primary_company", "primary_country",
             "director", "collection_name"]
MULTI_COLS = ["genres", "production_companies", "production_countries", "top_cast"]   # multi-valued, '|' separated

data_dictionary = pd.DataFrame([
    ("id", "Identifier", "int", "TMDB movie ID. Identifier only; never a predictive feature"),
    ("imdb_id", "Identifier", "str", "IMDb ID. Identifier only"),
    ("title / original_title", "Text", "str", "Movie titles. Identification only"),
    ("overview / tagline", "Text", "str", "Free-text plot summary and tagline (usable for TF-IDF)"),
    ("release_date", "Date", "datetime", "Release date"),
    ("release_year / month / dow", "Numeric (date-derived)", "int", "Year, month (1-12), weekday (0=Mon) extracted from release_date"),
    ("budget", "Numeric", "int (USD)", "Production budget. TMDB stores unknown values as 0"),
    ("revenue", "Numeric (TARGET)", "int (USD)", "Box-office revenue. 0 means unknown"),
    ("runtime", "Numeric", "float (min)", "Length in minutes. 0 means unknown"),
    ("popularity", "Numeric", "float", "TMDB popularity score at time of collection (changes daily)"),
    ("vote_average / vote_count", "Numeric", "float / int", "Audience rating (0-10) and number of votes (available only after release)"),
    ("n_genres / n_companies / n_countries / n_spoken_languages", "Numeric (derived)", "int", "Counts extracted from nested fields"),
    ("cast_size / crew_size", "Numeric (derived)", "int", "Number of credited cast / crew"),
    ("status", "Categorical", "str", "Released, Post Production, ..."),
    ("original_language", "Categorical", "str", "ISO 639-1 language code"),
    ("primary_genre / primary_company / primary_country", "Categorical (derived)", "str", "First listed value of the nested list"),
    ("director", "Categorical (derived)", "str", "Crew member whose job is Director (first)"),
    ("collection_name", "Categorical", "str", "Franchise / collection name (NaN if standalone)"),
    ("genres / production_companies / production_countries / top_cast", "Multi-valued categorical", "str ('|' separated)", "Need splitting/multi-hot encoding by Person 2"),
    ("adult / video", "Boolean", "bool", "Adult flag, video flag"),
    ("has_homepage / has_tagline / in_collection", "Boolean (derived)", "bool", "Presence flags derived from nested/empty fields"),
], columns=["Attribute", "Kind", "Type", "Meaning"])
display(data_dictionary)

# sanity check: every column is classified exactly once
classified = set(ID_COLS+TEXT_COLS+DATE_COLS+BOOL_COLS+NUM_COLS+CAT_COLS+MULTI_COLS)
print("Unclassified columns:", set(df.columns) - classified, "| Not in data:", classified - set(df.columns))

**What was done:** classified every column as identifier, text, date, Boolean, numerical, categorical or multi-valued categorical, and documented the meaning and data type of each attribute in a data dictionary.
**Why:** the brief requires every selected attribute to be documented and the variable types to be identified. This classification also drives later steps: numeric columns get scaling, categorical columns get encoding, and ID/text columns are excluded from numeric features.
**Output:** the data dictionary and a check that no column was left unclassified.
**Observation:** the dataset mixes numerical, categorical, date-derived, Boolean and multi-valued fields, as the brief requires. `popularity`, `vote_average` and `vote_count` are post-release audience statistics, so Person 2 must judge whether they can be used as inputs. They would not exist at the intended prediction time (before release) and may leak information about revenue.

In [ ]:
display(df[NUM_COLS].describe().T)
display(df[BOOL_COLS].describe().T)

**What was done:** generated descriptive statistics (count, mean, std, min, quartiles, max) for numerical and Boolean variables.
**Why:** summary statistics reveal scale, spread, impossible values and signs of skewness. They also confirm the 0-as-missing problem, since `min` is 0 for budget, revenue and runtime.
**Output:** two summary tables.
**Observation:** *(complete after running)* — compare mean vs median (50%) for `budget`, `revenue`, `vote_count`. A mean much larger than the median indicates strong right skew (confirmed formally in Person 2's skewness step). Note that `min = 0` for budget/revenue/runtime, which is TMDB's "unknown" code.

In [ ]:
for c in ["status", "original_language", "primary_genre", "primary_country"]:
    vc = df[c].value_counts(dropna=False)
    print(f"--- {c}: {df[c].nunique()} unique ---")
    display(vc.head(10).to_frame("count").assign(percent=lambda x: (x["count"]/len(df)*100).round(1)))

# multi-valued genres: split and count
genre_counts = df["genres"].dropna().str.split("|").explode().value_counts()
print("--- genres (multi-valued, exploded) ---"); display(genre_counts.to_frame("count"))

print("Unique-value counts for high-cardinality fields:")
print(df[["director", "primary_company", "collection_name"]].nunique())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
genre_counts.plot.barh(ax=ax[0], color="steelblue").invert_yaxis(); ax[0].set_title("Movies per genre (multi-valued)")
df["original_language"].value_counts().head(10).plot.bar(ax=ax[1], color="darkorange"); ax[1].set_title("Top 10 original languages")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "p1_categorical_frequencies.png"), dpi=120); plt.show()

**What was done:** computed unique values and frequency distributions for the categorical variables (status, language, genre, country), exploded the multi-valued `genres` field, and counted the unique values in the high-cardinality fields (director, company, collection). The genre and language frequencies were then plotted.
**Why:** the number of categories decides the encoding strategy. Low-cardinality fields (status, language) suit one-hot encoding. High-cardinality fields (director, company) would create hundreds of columns, so they need grouping or target encoding. The multi-valued genre field must be split before encoding.
**Output:** frequency tables and two bar charts.
**Observation:** *(complete after running)* — expect English to dominate `original_language`, and Drama/Action/Comedy to lead the genres. Report the number of unique directors and companies. A value close to the row count means those columns are close to identifiers and are poor candidates for plain one-hot encoding.

## Step 7 — Missing-value analysis (original condition)
TMDB stores **unknown** budget, revenue and runtime as `0`. A zero budget is not a real budget, so for reporting these zeros are treated as missing. The brief also says records with a missing **target** must be excluded from model training and evaluation.

In [ ]:
ZERO_AS_MISSING = ["budget", "revenue", "runtime"]

report = pd.DataFrame({
    "NaN_count": df.isna().sum(),
    "zero_count": [(df[c] == 0).sum() if c in ZERO_AS_MISSING else 0 for c in df.columns],
})
report["missing_total"] = report["NaN_count"] + report["zero_count"]
report["missing_%"] = (report["missing_total"] / len(df) * 100).round(2)
display(report.sort_values("missing_%", ascending=False).query("missing_total > 0"))

df_zero = df.copy()
df_zero[ZERO_AS_MISSING] = df_zero[ZERO_AS_MISSING].replace(0, np.nan)
print("Rows with missing target (revenue):", df_zero["revenue"].isna().sum(),
      f"({df_zero['revenue'].isna().mean()*100:.1f}%)")

**What was done:** counted real `NaN`s and `0`s in budget/revenue/runtime for every column, combined them into the total missing count and percentage, and then converted the zeros to `NaN` in a separate copy (`df_zero`).
**Why:** the brief requires the *actual* original missing counts and percentages, and says that TMDB's 0 means "unknown". Reporting only `isna()` would show almost no missing data and hide the real problem.
**Output:** a table of columns with missing values, and the number of rows whose target is missing.
**Observation:** *(complete after running)* — budget and revenue should show the largest missing shares (zeros), `tagline` and `collection_name` have natural gaps, and the rest have little or none. The rows with missing revenue are excluded from modelling in Step 9.

In [ ]:
# Is budget/revenue "unknown" random, or linked to era / language? (a quick missingness-pattern check)
tmp = df_zero.assign(budget_missing=df_zero["budget"].isna(), revenue_missing=df_zero["revenue"].isna(),
                     is_english=np.where(df_zero["original_language"] == "en", "English", "Non-English"))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
(tmp.groupby("release_year")[["budget_missing", "revenue_missing"]].mean() * 100).plot.bar(ax=ax[0])
ax[0].set_ylabel("% missing"); ax[0].set_title("Missing budget / revenue by release year")
(tmp.groupby("is_english")[["budget_missing", "revenue_missing"]].mean() * 100).plot.bar(ax=ax[1], rot=0)
ax[1].set_ylabel("% missing"); ax[1].set_title("Missing budget / revenue: English vs non-English")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "p1_missing_pattern.png"), dpi=120); plt.show()

**What was done:** compared the share of missing budget/revenue across release years and between English and non-English films.
**Why:** if missingness depends on year or language, it is *not* missing completely at random, and simply dropping or mean-filling the rows would bias the data. This evidence also justifies the imputation choices later.
**Output:** two bar charts.
**Observation:** *(complete after running)* — state whether missing rates rise for recent years (2025–2026 films have had little time to earn revenue) and whether non-English films have higher rates. If so, the missingness is systematic and that must be written in the report.

## Step 8 — Duplicate detection and removal
Duplicates are checked at three levels: (1) exact duplicate rows, (2) repeated `id` values, (3) the same title with the same release date under different IDs (a "possible duplicate", reported for review but not automatically removed).
Duplicates are handled **before** the artificial missingness is created, so that masked cells are never lost when a row is deleted.

In [ ]:
n_before = len(df_zero)
exact_dups = df_zero.duplicated().sum()
id_dups = df_zero["id"].duplicated().sum()
title_date_dups = df_zero.duplicated(subset=["title", "release_date"], keep=False).sum()
print(f"Exact duplicate rows      : {exact_dups}")
print(f"Repeated movie IDs        : {id_dups}")
print(f"Same title + release date : {title_date_dups} rows (possible duplicates, manual review)")
if title_date_dups:
    display(df_zero[df_zero.duplicated(subset=["title", "release_date"], keep=False)][["id", "title", "release_date", "vote_count"]].sort_values("title"))

df_final = df_zero.drop_duplicates().drop_duplicates(subset="id", keep="first").reset_index(drop=True)
print(f"\nRows before: {n_before} | removed: {n_before - len(df_final)} | rows after: {len(df_final)}")
assert df_final["id"].is_unique
df_final.to_csv(os.path.join(DATA_DIR, "tmdb_movies_final.csv"), index=False)

**What was done:** counted exact duplicate rows, repeated IDs and same-title/same-date pairs, removed exact and repeated-ID duplicates (keeping the first), asserted ID uniqueness, and saved the result as `tmdb_movies_final.csv`. In this file the zeros in budget/revenue/runtime are already converted to `NaN`.
**Why:** duplicate movies would be counted twice in training and could appear in both the train and test sets, which inflates test scores (data leakage). The collection used a global `seen` set, so zero duplicates is the expected result.
**Output:** the three duplicate counts, rows removed, and the final row count.
**Observation:** *(complete after running)* — report the numbers found and removed. If both are 0, the de-duplication logic in the collection step worked. Same-title pairs, if any, are usually different films that share a title (remakes), so they are kept.

## Step 9 — Excluding missing targets and introducing ~5% artificial missing values
1. **Model base:** rows with a missing target (`revenue`) are removed, because they cannot be used for training or evaluation.
2. **Working copy:** about 5% of the values in selected **input-feature** columns are set to `NaN` at random (MCAR), using the fixed seed. The target is never touched.
3. The untouched copy is kept as `truth`, so the imputation quality can later be measured against the real values (used for the advanced technique).

Only cells that currently hold a value are masked, so each chosen column gets about 5% *additional* missingness. Budget and runtime already carry their natural missing values from the zeros.

In [ ]:
df_model = df_final[df_final["revenue"].notna()].reset_index(drop=True)
print(f"Rows kept for modelling: {len(df_model)} (dropped {len(df_final)-len(df_model)} with missing target)")
df_model.to_csv(os.path.join(DATA_DIR, "tmdb_movies_model_base.csv"), index=False)

MASK_NUM = ["budget", "runtime", "popularity", "vote_count", "vote_average"]
MASK_CAT = ["original_language", "primary_genre", "primary_country"]
MASK_COLS = MASK_NUM + MASK_CAT
MISSING_FRACTION = 0.05

rng = np.random.default_rng(SEED)
work = df_model.copy()
truth = df_model.copy()
mask = pd.DataFrame(False, index=work.index, columns=MASK_COLS)
for c in MASK_COLS:
    available = work.index[work[c].notna()]
    k = min(int(round(MISSING_FRACTION * len(work))), len(available))
    chosen = rng.choice(available, size=k, replace=False)
    work.loc[chosen, c] = np.nan
    mask.loc[chosen, c] = True

assert work["revenue"].notna().all(), "Target must never contain missing values"
before = df_model[MASK_COLS].isna().sum()
after = work[MASK_COLS].isna().sum()
summary = pd.DataFrame({"missing_before": before, "artificially_added": mask.sum(), "missing_after": after,
                        "missing_after_%": (after / len(work) * 100).round(2)})
display(summary)
print(f"Artificially masked cells: {mask.values.sum()} of {len(work)*len(MASK_COLS)} selected cells "
      f"({mask.values.sum()/(len(work)*len(MASK_COLS))*100:.2f}%)")
work.to_csv(os.path.join(DATA_DIR, "tmdb_movies_working_missing.csv"), index=False)

**What was done:** removed the rows with missing revenue (the target), then created a separate working copy and randomly blanked about 5% of the values in eight input columns (five numerical, three categorical). An assertion confirms the target has no missing values. The working copy is saved as `tmdb_movies_working_missing.csv`.
**Why:** the brief requires a controlled ~5% missingness in input features, on a separate copy and never in the target, so that imputation techniques can be demonstrated. Keeping `truth` allows an honest accuracy check of the imputation.
**Output:** a table showing missing counts before, the added missing values, and the totals after masking, plus the overall percentage of masked cells.
**Observation:** *(complete after running)* — the overall masked share should be about 5% of the selected cells. `budget` and `runtime` show more than 5% in total because they already contained natural missing values from the zeros.

## Step 10 — Imputation (baseline) and missing-value comparison
**Baseline strategy, chosen by variable type and distribution:**
- Numerical, skewed (budget, popularity, vote_count): **median** (robust to extreme blockbusters).
- Numerical, roughly symmetric (vote_average, runtime): median is also fine (the mean would be nearly equal).
- Categorical: **mode** (most frequent category).

In [ ]:
from scipy.stats import skew
print("Skewness of numeric columns (original, non-missing values):")
print(df_model[MASK_NUM].apply(lambda s: skew(s.dropna())).round(2).to_string())

simple = work.copy()
for c in MASK_NUM:
    simple[c] = simple[c].fillna(simple[c].median())
for c in MASK_CAT:
    simple[c] = simple[c].fillna(simple[c].mode()[0])

cmp = pd.DataFrame({"before_imputation": work[MASK_COLS].isna().sum(), "after_imputation": simple[MASK_COLS].isna().sum()})
display(cmp)

**What was done:** measured skewness to choose the strategy, imputed numerical columns with the median and categorical columns with the mode, and compared missing counts before and after.
**Why:** the mean is pulled by extreme values in right-skewed variables such as budget and vote count, so the median is the safer single-value fill. The mode is the only meaningful single-value fill for categories. This also forms the **baseline** against which the advanced technique is compared.
**Output:** skewness values and a before/after missing-count table (all zeros after imputation).
**Observation:** *(complete after running)* — large positive skewness in budget, popularity and vote_count justifies the median. The limitation of this approach is that every missing value gets the *same* number, which shrinks variance and ignores relationships between features. This motivates Step 11.

## Step 11 — Advanced technique #1: KNN and Iterative imputation

**Why this technique (evidence from our data):** budget, popularity, vote count and runtime are clearly related, because big-budget films tend to attract more votes and higher popularity. A single median ignores this. KNN imputation (`KNNImputer`) fills each gap from the 5 most similar films, and Iterative imputation (`IterativeImputer`) predicts each missing value from the other columns.

**Effect on the data:** it changes the features that the models will use (the cleaned input dataset), which satisfies the brief's requirement that at least one technique must change the data or features.

**Design choices (each can be defended in the evaluation):**
- Skewed columns (budget, popularity, vote_count) are log-transformed **inside the imputer** only, and the output is transformed back, so distances are not dominated by billion-dollar outliers. Columns are standardised before KNN because KNN is distance-based.
- The **target (`revenue`) is not used** to impute the inputs. Using it would leak target information into the features.
- Quality is measured on the artificially masked cells, where the true values are known (`truth`), so the before-and-after comparison is evidence rather than opinion.
- Fit and apply are separate functions so Person 2 can **fit on the training set only** and apply to the test set, as Section 10 of the brief requires.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler

IMP_FEATS = ["budget", "runtime", "popularity", "vote_count", "vote_average",
             "n_genres", "n_companies", "cast_size", "release_year"]      # NOTE: revenue deliberately excluded
LOG_FEATS = ["budget", "popularity", "vote_count"]
ROUND_FEATS = ["n_genres", "n_companies", "cast_size", "release_year"]

def _to_space(df):
    X = df[IMP_FEATS].astype(float).copy()
    for c in LOG_FEATS:
        X[c] = np.log1p(X[c])
    return X

def fit_num_imputer(train_df, method="knn", k=5):
    scaler = StandardScaler().fit(_to_space(train_df))
    imp = (KNNImputer(n_neighbors=k, weights="distance") if method == "knn"
           else IterativeImputer(max_iter=15, random_state=SEED, sample_posterior=False))
    imp.fit(scaler.transform(_to_space(train_df)))
    return scaler, imp

def apply_num_imputer(df, state):
    scaler, imp = state
    Z = imp.transform(scaler.transform(_to_space(df)))
    X = pd.DataFrame(scaler.inverse_transform(Z), columns=IMP_FEATS, index=df.index)
    for c in LOG_FEATS:
        X[c] = np.expm1(X[c]).clip(lower=0)
    X["runtime"] = X["runtime"].clip(lower=1)
    X["vote_average"] = X["vote_average"].clip(0, 10)
    X[ROUND_FEATS] = X[ROUND_FEATS].round()
    out = df.copy()
    for c in MASK_NUM:
        out[c] = df[c].fillna(X[c])
    return out

knn_state = fit_num_imputer(work, "knn")
iter_state = fit_num_imputer(work, "iterative")
imp_knn = apply_num_imputer(work, knn_state)
imp_iter = apply_num_imputer(work, iter_state)
for d in (imp_knn, imp_iter):
    for c in MASK_CAT:                       # categoricals: mode for both
        d[c] = d[c].fillna(work[c].mode()[0])
print("Remaining missing in input columns -> KNN:", int(imp_knn[MASK_COLS].isna().sum().sum()),
      "| Iterative:", int(imp_iter[MASK_COLS].isna().sum().sum()))

**What was done:** wrote `fit_num_imputer()` and `apply_num_imputer()` (log-transform → standardise → KNN or Iterative imputation → inverse-transform → clip to valid ranges). Both imputers were fitted on the working copy and applied to it. Categorical columns use the mode.
**Why:** this builds the advanced imputation as a reusable, leak-free component, with a fit step and a separate apply step, so Person 2 can refit it on the training split only. Clipping keeps imputed values physically valid (non-negative budget, rating between 0 and 10, runtime of at least 1).
**Output:** confirmation that no missing input values remain.
**Observation:** both imputers filled every artificial gap. The next cell measures *how good* the fills are.

In [ ]:
def imputation_error(imputed):
    rows = []
    for c in MASK_NUM:
        m = mask[c]
        err = imputed.loc[m, c] - truth.loc[m, c]
        rows.append({"column": c, "MAE": err.abs().mean(), "RMSE": np.sqrt((err ** 2).mean()),
                     "normalised_MAE": err.abs().mean() / truth[c].std()})
    return pd.DataFrame(rows).set_index("column")

err_tbl = pd.concat({"Median (baseline)": imputation_error(simple),
                     "KNN (k=5)": imputation_error(imp_knn),
                     "Iterative": imputation_error(imp_iter)}, axis=1)
display(err_tbl.round(3))

norm = pd.DataFrame({m: err_tbl[m]["normalised_MAE"] for m in ["Median (baseline)", "KNN (k=5)", "Iterative"]})
print("Normalised MAE (MAE / std of true values) - lower is better:")
display(norm.round(3).assign(best=norm.idxmin(axis=1)))
print("Mean normalised MAE:\n", norm.mean().round(3).to_string())

# categorical: accuracy of mode-fill on the masked cells
acc = {c: (simple.loc[mask[c], c] == truth.loc[mask[c], c]).mean() for c in MASK_CAT}
print("\nMode-imputation accuracy on masked categorical cells:", {k: round(v, 3) for k, v in acc.items()})

norm.plot.bar(figsize=(9, 4), rot=0); plt.ylabel("Normalised MAE (lower = better)")
plt.title("Imputation error on artificially masked cells"); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "p1_imputation_error.png"), dpi=120); plt.show()

**What was done:** compared median, KNN and Iterative imputation on the cells that were artificially masked. Because the true values are known, MAE, RMSE and a scale-free normalised MAE (MAE divided by the standard deviation) could be computed per column. Mode accuracy was checked for the categorical columns.
**Why:** the brief requires a before-and-after comparison for each advanced technique. Normalised MAE puts budget (millions of dollars) and vote_average (0–10) on a common scale, so the methods can be ranked fairly.
**Output:** the full error table, the normalised table with the best method per column, the mean normalised MAE, categorical accuracy, and a bar chart.
**Observation:** *(complete after running)* — state which method wins on average and for which columns. A typical result is that KNN/Iterative beat the median clearly for `vote_count`, `popularity` and `budget`, because those are related to each other, while the gain for `vote_average` is small. Quote the mean normalised MAE of each method and the percentage improvement over the median. If the median wins for some column, say so honestly. A result that goes against expectations still counts as a finding. Note that mode-imputation accuracy for categories is limited, and that the dominant class (English / Drama / United States) drives it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
for ax, col in zip(axes, ["budget", "vote_count", "popularity"]):
    sns.kdeplot(np.log1p(truth[col].dropna()), ax=ax, label="True (before masking)", color="black", lw=2)
    sns.kdeplot(np.log1p(simple[col]), ax=ax, label="Median-imputed", color="tab:red")
    sns.kdeplot(np.log1p(imp_knn[col]), ax=ax, label="KNN-imputed", color="tab:green")
    ax.set_title(f"log(1+{col})"); ax.set_xlabel("")
axes[0].legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "p1_imputation_distributions.png"), dpi=120); plt.show()

std_tbl = pd.DataFrame({"true_std": truth[MASK_NUM].std(), "median_std": simple[MASK_NUM].std(), "knn_std": imp_knn[MASK_NUM].std()})
display(std_tbl.round(2))

**What was done:** plotted the distribution of `budget`, `vote_count` and `popularity` (log scale) for the true data, the median-imputed data and the KNN-imputed data, and compared the standard deviations.
**Why:** a single-value fill creates an artificial spike at the median and shrinks the variance. This is the "before and after on the data" evidence the brief asks for.
**Output:** three density plots and a standard-deviation table.
**Observation:** *(complete after running)* — expected: the red median curve has a sharp spike near the middle, while the green KNN curve follows the black true curve more closely. The KNN standard deviations should be closer to the true standard deviations than the median ones. Say whether this held in your run.

## Step 12 — Final cleaned dataset, documentation and hand-over to Person 2
The KNN-imputed dataset is saved as the cleaned dataset (the choice of KNN over the median is justified by the comparison above; switch to `imp_iter` if Iterative won clearly in your run).

In [ ]:
final_imputed = imp_knn.copy()
final_imputed.to_csv(os.path.join(DATA_DIR, "tmdb_movies_imputed.csv"), index=False)

manifest = {p: sha256_file(os.path.join(DATA_DIR, p)) for p in
            ["tmdb_movies_raw.csv", "raw_responses.jsonl", "movie_ids.csv", "tmdb_movies_final.csv",
             "tmdb_movies_model_base.csv", "tmdb_movies_working_missing.csv", "tmdb_movies_imputed.csv"]
            if os.path.exists(os.path.join(DATA_DIR, p))}
ids_fingerprint = hashlib.sha256(",".join(map(str, sorted(pd.read_csv(os.path.join(DATA_DIR, "movie_ids.csv"))["id"]))).encode()).hexdigest()
manifest["movie_ids_list"] = ids_fingerprint
with open(os.path.join(DATA_DIR, "sha256_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

pipeline = pd.DataFrame([
    ("Raw collected rows", len(df_raw)),
    ("After duplicate removal (tmdb_movies_final.csv)", len(df_final)),
    ("Rows with missing target removed", len(df_final) - len(df_model)),
    ("Rows available for modelling", len(df_model)),
    ("Artificially masked cells", int(mask.values.sum())),
    ("Missing input cells after KNN imputation", int(final_imputed[MASK_COLS].isna().sum().sum())),
], columns=["Stage", "Value"])
display(pipeline)
print("Files in data/:", sorted(os.listdir(DATA_DIR)))
print("\nSHA-256 manifest updated.")

**What was done:** saved the final imputed dataset, refreshed the SHA-256 manifest for all deliverable files, and printed a stage-by-stage row summary.
**Why:** the brief requires the raw dataset, final CSV, working dataset with missing values and a cleaned dataset or reproducible pipeline. The manifest proves which exact versions were produced.
**Output:** the pipeline summary table and the file list.
**Observation:** *(complete after running)* — record the final modelling row count. Rows drop from the raw count to the modelling count mainly because many movies have unknown revenue.

### Hand-over notes for Person 2
- Start from `tmdb_movies_model_base.csv` (no duplicates, 0→NaN done, target present). For the missing-value-handled version use `tmdb_movies_imputed.csv`.
- **Target:** `revenue`. **Do not** use `id`, `imdb_id`, `title`, `original_title` as numeric predictors.
- **Leakage warning:** `popularity`, `vote_average`, `vote_count` exist only after release. Decide and justify whether they are valid inputs for the intended prediction point.
- Multi-valued columns (`genres`, `production_companies`, `production_countries`, `top_cast`) are `|`-separated and need splitting/multi-hot encoding. `director` and `primary_company` have very high cardinality.
- When building the train/test pipeline, **refit** `fit_num_imputer()` on the training split only and call `apply_num_imputer()` on the test split. This notebook fit on the full working copy only to demonstrate and compare the method.
- Person 2's advanced technique (#2) must be a different technique from KNN/Iterative imputation.

### Limitations to cite in the report (Person 1 part)
1. Sorting by vote count favours popular films, so the sample is not a random sample of all movies.
2. Budget and revenue are crowd-sourced and often missing or inaccurate. Zeros are treated as missing, but a small number may be true low-budget values.
3. `popularity` is a snapshot at the collection date and changes daily, so re-collection will give slightly different values.
4. 2026 is a partial year, so films released recently may not yet have final revenue.
5. The artificial missingness is MCAR (random), which is easier to impute than the real missingness, which is likely systematic (see Step 7).